# 🧪 Diseño Experimental Factorial $2^3$ y Comparativa $2^4$ vs $2^{4-1}$ Fraccionado (E2)

**Proyecto Integrador — Trusted Data Lab (Etapa E2)**  
**Integrantes:** Juan Antonio Ortiz Hinojosa & Diego Alberto Virgen Cuéllar

---

## 🎯 Contenido del Notebook
1. **Diseño Factorial $2^3$ Aleatorizado con $r=3$ Réplicas ($N=24$ corridas)** sobre el rendimiento del pipeline:
   - Factor A: Tamaño de Lote (`-1`: 250 regs, `+1`: 1,500 regs)
   - Factor B: Estrategia de Validación (`-1`: Iteración Fila-a-Fila, `+1`: Vectorizado Pandera)
   - Factor C: Formato de Salida (`-1`: JSON Texto, `+1`: CSV Columnar)
2. **Tabla ANOVA con Efectos Principales e Interacciones** ($A, B, C, AB, AC, BC, ABC$).
3. **Validación de Supuestos** (Normalidad de Shapiro-Wilk y Homocedasticidad de Levene) y **Corrida de Confirmación**.
4. **Comparación Teórico-Práctica de Matriz $2^4$ Completa vs $2^{4-1}$ Fraccionada (Resolución IV, Generador $I=ABCD$)**.

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd

ROOT_DIR = Path.cwd().resolve()
if not (ROOT_DIR / "src").exists():
    ROOT_DIR = ROOT_DIR.parent.parent
sys.path.insert(0, str(ROOT_DIR))

from src.analytics_e3_e4 import run_factorial_experiment_2k

res_e2 = run_factorial_experiment_2k()
display(res_e2["experimento_2_3_df"].head(12))

,corrida_estandar,replica,A_tamano_lote,B_validacion_vectorizada,C_formato_columnar,A_etiqueta,B_etiqueta,C_etiqueta,combinacion,tiempo_ejecucion_ms,throughput_regs_por_seg,orden_aleatorio
0,10,2,-1,-1,1,250_regs (-1),Fila_a_Fila (-1),CSV_Optimo (+1),A=- B=- C=+,214.04,4672.0,1
1,8,1,1,1,1,1500_regs (+1),Pandera_Vector (+1),CSV_Optimo (+1),A=+ B=+ C=+,112.29,8905.5,2
2,6,1,1,-1,1,1500_regs (+1),Fila_a_Fila (-1),CSV_Optimo (+1),A=+ B=- C=+,197.27,5069.2,3
3,21,3,1,-1,-1,1500_regs (+1),Fila_a_Fila (-1),JSON_Crudo (-1),A=+ B=- C=-,229.87,4350.3,4
4,24,3,1,1,1,1500_regs (+1),Pandera_Vector (+1),CSV_Optimo (+1),A=+ B=+ C=+,111.47,8971.0,5
5,7,1,1,1,-1,1500_regs (+1),Pandera_Vector (+1),JSON_Crudo (-1),A=+ B=+ C=-,150.20,6657.8,6
6,9,2,-1,-1,-1,250_regs (-1),Fila_a_Fila (-1),JSON_Crudo (-1),A=- B=- C=-,232.48,4301.4,7
7,2,1,-1,-1,1,250_regs (-1),Fila_a_Fila (-1),CSV_Optimo (+1),A=- B=- C=+,217.25,4603.0,8
8,14,2,1,-1,1,1500_regs (+1),Fila_a_Fila (-1),CSV_Optimo (+1),A=+ B=- C=+,202.70,4933.4,9
9,12,2,-1,1,1,250_regs (-1),Pandera_Vector (+1),CSV_Optimo (+1),A=- B=+ C=+,142.91,6997.4,10


## 1. Tabla ANOVA Multifactorial ($2^3$), Supuestos y Corrida de Confirmación

In [2]:
display(res_e2["anova_tabla_df"])
print("\n=== Validación de Supuestos del ANOVA ===")
print(json.dumps(res_e2["supuestos_anova"], indent=2))
print("\n=== Corrida de Confirmación ===")
print(json.dumps(res_e2["corrida_confirmacion"], indent=2))

,Fuente_Variacion,Efecto_Estimado_ms,Suma_Cuadrados_SS,GL,Cuadrado_Medio_MS,F_ Estadistico,p_valor,Significativo_alpha_05,Contribucion_Pct
0,A (Tamaño de Lote),-24.00,3455.28,1,3455.28,299.74,0.000000,True,8.67
1,B (Validación Vectorizada Pandera),-69.65,29110.22,1,29110.22,2525.25,0.000000,True,73.07
2,C (Formato Salida Optimizado),-30.46,5566.57,1,5566.57,482.89,0.000000,True,13.97
3,AB (Interacción Lote × Validación),-14.02,1179.22,1,1179.22,102.29,0.000000,True,2.96
4,AC (Interacción Lote × Formato),-3.01,54.51,1,54.51,4.73,0.045009,True,0.14
5,BC (Interacción Validación × Formato),-6.57,259.06,1,259.06,22.47,0.000222,True,0.65
6,ABC (Interacción Triple),2.15,27.67,1,27.67,2.40,0.140859,False,0.07
7,Error Puro (Réplicas),0.00,184.44,16,11.53,0.00,1.000000,False,0.46



=== Validación de Supuestos del ANOVA ===
{
  "shapiro_wilk_stat": 0.9616,
  "shapiro_wilk_pvalue": 0.4711,
  "normalidad_cumplida": true,
  "levene_stat": 0.7698,
  "levene_pvalue": 0.6206,
  "homocedasticidad_cumplida": true
}

=== Corrida de Confirmación ===
{
  "configuracion_optima": "A=+1 (Lote 1500), B=+1 (Pandera Vectorizado), C=+1 (CSV Columnar)",
  "tiempo_medio_base_ms": 233.8,
  "tiempo_medio_optimo_ms": 111.84,
  "reduccion_latencia_pct": 52.17
}


## 2. Comparativa Diseño $2^4$ Completo (16 corridas base) vs Fracción $2^{4-1}$ ($I=ABCD$, 8 corridas base)

In [3]:
comp = res_e2["comparativa_2_4_vs_2_4_1"]
print("Generador:", comp["generador_fraccion"])
print("Resolución:", comp["resolucion"])
print("Ahorro Experimental (%):", comp["ahorro_experimental_pct"])
print("Estructura de Alias:", comp["estructura_alias_principales"])
display(comp["matriz_fraccionada_2_4_1_df"])

Generador: I = A B C D  (D = A * B * C)
Resolución: Resolución IV (Ningún efecto principal está confundido con otro efecto principal ni con interacciones de 2 factores)
Ahorro Experimental (%): 50.0
Estructura de Alias: ['[A] = A + BCD', '[B] = B + ACD', '[C] = C + ABD', '[D] = D + ABC', '[AB] = AB + CD', '[AC] = AC + BD', '[AD] = AD + BC']


,A_Lote,B_Vectorizacion,C_Formato,D_Cache_Memoria
0,-1,-1,-1,-1
1,-1,-1,1,1
2,-1,1,-1,1
3,-1,1,1,-1
4,1,-1,-1,1
5,1,-1,1,-1
6,1,1,-1,-1
7,1,1,1,1
